# Lipid Droplet Quantification: BODIPY + Perilipin (PLIN1) Co-localization

This notebook quantifies lipid droplets in adipose tissue sections imaged by Lunaphore multiplex fluorescence at 20× (qPTIFF format), comparing **Day 1** vs **Day 60** post-treatment across matched patients.

**Channels used:**
| Channel | Fluorophore | Target |
|---------|-------------|--------|
| FITC | BODIPY | Lipid droplet content (green) |
| Cy5 | Perilipin / PLIN1 | Adipocyte membrane marker (magenta) |
| DAPI | DAPI | Nuclei |
| ATTO 550 | — | Autofluorescence / background |

**Analysis pipeline:**
1. Parse channel metadata from qPTIFF headers
2. Identify and crop individual tissue sections interactively
3. Segment lipid droplets (BODIPY): Otsu threshold → Perilipin-guided watershed
4. Classify adipocytes: size filter + Perilipin co-localization (≥ 10 px overlap)
5. Load QuPath-derived tissue areas (adipose tissue segmentation done in QuPath)
6. Compute per-section metrics: droplet count density and lipid area fraction
7. Statistical comparison: Day1 vs Day60 (t-test, patient-colored strip plots)

> **Note on tissue area:** Tissue areas were obtained by thresholding segmentation in [QuPath](https://qupath.github.io/) and are hardcoded in Section 5.

## Imports

In [ ]:
import glob
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator

import tifffile as tff
from xml.etree import ElementTree
from scipy import ndimage as ndi
from skimage import filters, measure, morphology, color
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

mpl.rcParams['pdf.fonttype'] = 42

## Configuration

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
QPTIFF_DIR   = '../data/bodipy_qtiffs/'        # directory containing {slide}_FITC_20x.qptiff
CROP_CSV     = '../data/crop_idx_lv0.csv'      # section crop coordinates (output of Section 2)
RESULTS_DIR  = '../results/'

# ── Sample lists ─────────────────────────────────────────────────────────────
SLIST = ['Slide1', 'Slide2', 'Slide3', 'Slide4', 'Slide5', 'Slide6']

# ── Sample metadata: case ID → timepoint and patient ID ─────────────────────
DICT_CID2TYPE = {
    '1622753': 'Day60', '1629501': 'Day60', '1629507': 'Day60',
    '1622930': 'Day1',  '1621945': 'Day1',  '1622668': 'Day60',
    '1623464': 'Day60', '1615778': 'Day1',  '1615782': 'Day1',
    '1631298': 'Day60', '1623790': 'Day1',  '1614806': 'Day1',
}
DICT_CID2PID = {
    '1622753': 4, '1629501': 6, '1629507': 10,
    '1622930': 6, '1621945': 7, '1622668': 3,
    '1623464': 2, '1615778': 2, '1615782': 3,
    '1631298': 9, '1623790': 5, '1614806': 4,
}

# ── Segmentation parameters ───────────────────────────────────────────────────
MIN_AREA      = 500     # minimum droplet area in pixels² (removes dust/noise)
MAX_AREA      = 50000   # maximum droplet area in pixels² (removes tissue artifacts)
MIN_OVERLAP   = 10      # minimum Perilipin pixel overlap to call a droplet an adipocyte
PIXEL_SIZE_UM = 0.4995  # microns per pixel at 20×

# Slides with non-standard brightness requiring manual threshold (255 - value)
MANUAL_THRESH = {
    'Slide6': {'thr_b': 135, 'thr_p': 10},
    # 'Slide3': {'thr_b': 35, 'thr_p': 15},  # uncomment if needed
}
DO_NOISE_FILTER = False  # apply median + white tophat to Perilipin channel

# ── Plot aesthetics ───────────────────────────────────────────────────────────
DICT_COLOR    = {'Day1': '#FAB31B', 'Day60': '#4766B0'}
HUE_ORDER     = ['Day1', 'Day60']
DICT_PIDCOLOR = {
    '7': (0.40, 0.76, 0.65), '4': (0.99, 0.55, 0.38),
    '5': (0.55, 0.63, 0.80), '2': (0.91, 0.54, 0.76),
    '3': (0.65, 0.85, 0.33), '6': (1.00, 0.85, 0.18),
    '10':(0.90, 0.77, 0.58), '9': (0.70, 0.70, 0.70),
}

---
## 1. Parse Channel Metadata

Read channel names from the qPTIFF XML metadata of any one slide to build the channel-name → index mapping used throughout the notebook.

In [ ]:
def parse_channel_info(qptiff_path):
    """
    Extract channel names and indices from Lunaphore qPTIFF metadata.

    Returns
    -------
    dict
        {channel_name: page_index}, e.g. {'DAPI': 0, 'FITC': 1, ...}
    """
    channel_info = {}
    with tff.TiffFile(qptiff_path) as tif:
        for i, page in enumerate(tif.series[0].pages):
            try:
                root = ElementTree.fromstring(page.description)
                channel_name = root.find('.//Name').text
                channel_info[channel_name] = i
                print(f"  Channel {i}: {channel_name}")
            except (ElementTree.ParseError, AttributeError):
                print(f"  Channel {i}: [metadata not found]")
    return channel_info

# Parse from any available slide (channel layout is identical across slides)
sample_path = f"{QPTIFF_DIR}{SLIST[0]}_FITC_20x.qptiff"
dict_channelinfo = parse_channel_info(sample_path)

# Map fluorophore names to biological targets
DICT_CH2TARGET = {
    'DAPI':    'DAPI',
    'FITC':    'BODIPY',
    'Cy5':     'Perilipin',
    'ATTO 550':'background',
}

CH_ID_BODIPY = dict_channelinfo['FITC']
CH_ID_PERL   = dict_channelinfo['Cy5']
CH_ID_DAPI   = dict_channelinfo['DAPI']
print(f"\nBODIPY channel index: {CH_ID_BODIPY}")
print(f"Perilipin channel index: {CH_ID_PERL}")

---
## 2. Identify & Crop Tissue Sections

Each qPTIFF slide contains multiple tissue sections. This section identifies non-background regions using connected component analysis on the BODIPY channel and prompts for a case ID for each section found.

> **Skip this section** if crop coordinates are already saved — load `CROP_CSV` directly in Section 3.

In [ ]:
def get_crop_coordinates(image, padding=50):
    """
    Find bounding boxes of non-background tissue sections.

    For each region larger than 1000 pixels in height, displays a preview
    and prompts the user to enter a case ID (or 'remove' to exclude it).

    Parameters
    ----------
    image : 2D np.ndarray
        Single-channel image (BODIPY channel), grayscale.
    padding : int
        Padding in pixels added around each bounding box.

    Returns
    -------
    pd.DataFrame
        Columns: section_id, x0, xf, y0, yf
    """
    labels  = measure.label(image > 0)
    regions = measure.regionprops(labels)

    crop_list = []
    for region in regions:
        y0, x0, yf, xf = region.bbox
        if yf - y0 <= 1000:
            continue

        y0 = max(0, y0 - padding);  yf = min(image.shape[0], yf + padding)
        x0 = max(0, x0 - padding);  xf = min(image.shape[1], xf + padding)

        fig, ax = plt.subplots(figsize=(5, 8))
        ax.imshow(image[y0:yf, x0:xf], cmap='gray_r')
        ax.set_title('Cropped section preview')
        plt.show()
        print(f"Bounding box: y=[{y0}:{yf}], x=[{x0}:{xf}]")

        cid = input("Enter case ID (or 'remove' to skip): ")
        crop_list.append({'section_id': cid, 'x0': x0, 'xf': xf, 'y0': y0, 'yf': yf})

    return pd.DataFrame(crop_list)

In [ ]:
# Run interactively to identify sections across all slides
# (Only needed once; results are saved to CROP_CSV)

all_coords = []
for s in SLIST:
    path = f"{QPTIFF_DIR}{s}_FITC_20x.qptiff"
    print(f"Processing {s}...")
    with tff.TiffFile(path) as tif:
        image_he = tif.series[0].levels[0].asarray()
    img_for_proc = image_he[0, :, :]   # use first channel for tissue detection

    df_crops = get_crop_coordinates(img_for_proc)
    df_crops['sample'] = s
    all_coords.append(df_crops)

final_df = pd.concat(all_coords).reset_index(drop=True)
final_df = final_df.query('section_id != "remove"').reset_index(drop=True)
final_df.to_csv(CROP_CSV, index=False)
print(final_df)

---
## 3. Load Section Crop Coordinates

Load the previously saved crop coordinate table. Any manual coordinate corrections (e.g. sections that were cropped incorrectly) are applied here.

In [ ]:
final_df = pd.read_csv(CROP_CSV, dtype={'section_id': str})

# Manual corrections for any sections with incorrect auto-detected boundaries
# (add rows here as needed)
manual_corrections = [
    {'section_id': '1614806', 'x0': 7630,  'xf': 13440, 'y0': 0,     'yf': 5810,  'sample': 'Slide2'},
    {'section_id': '1622668', 'x0': 0,     'xf': 9650,  'y0': 15790, 'yf': 24480, 'sample': 'Slide2'},
]
for row in manual_corrections:
    # Replace existing entry if section_id already present, else append
    mask = (final_df['section_id'] == row['section_id']) & (final_df['sample'] == row['sample'])
    if mask.any():
        for k, v in row.items():
            final_df.loc[mask, k] = v
    else:
        final_df = pd.concat([final_df, pd.DataFrame([row])], ignore_index=True)

print(f"Total sections: {len(final_df)}")
print(final_df.to_string(index=False))

---
## 4. Lipid Droplet Segmentation & Adipocyte Classification

For each section, the pipeline:

1. **Threshold BODIPY** (Otsu, or manual threshold for slides with non-standard brightness)
2. **Threshold Perilipin** (Otsu, optionally with median denoising + white tophat for noisy slides)
3. **Subtract Perilipin from BODIPY mask** — Perilipin outlines adipocyte membranes and acts as a natural separator between adjacent droplets
4. **Watershed segmentation** on the cleaned BODIPY mask to split touching droplets
5. **Classify droplets as adipocytes** if they satisfy both:
   - Size within `[MIN_AREA, MAX_AREA]` pixels²
   - ≥ `MIN_OVERLAP` pixel overlap with the dilated Perilipin mask (confirms lipid droplet is bounded by adipocyte membrane)

**Visualization:** Each section produces a two-panel overlay:
- Left: rainbow-labeled individual droplets (watershed result)
- Right: green = confirmed adipocytes, magenta = Perilipin

In [ ]:
def segment_section(img_b, img_p, slide_name):
    """
    Segment lipid droplets and classify adipocytes for one tissue section crop.

    Parameters
    ----------
    img_b : 2D np.ndarray
        BODIPY channel crop.
    img_p : 2D np.ndarray
        Perilipin channel crop.
    slide_name : str
        Slide identifier, used to look up manual thresholds.

    Returns
    -------
    list of dict
        Per-droplet records with keys:
        area, is_adipocyte, is_real_size, is_on_membrane, centroid.
    label_b : 2D np.ndarray
        Watershed label image.
    overlay : 3D np.ndarray (H, W, 3)
        RGB overlay for visualization.
    rainbow_labels : 3D np.ndarray (H, W, 3)
        Rainbow-colored droplet label image.
    """
    # ── Thresholding ──────────────────────────────────────────────────────────
    if slide_name in MANUAL_THRESH:
        mask_b_raw = img_b > 255 - MANUAL_THRESH[slide_name]['thr_b']
        mask_p     = img_p > 255 - MANUAL_THRESH[slide_name]['thr_p']
    else:
        if DO_NOISE_FILTER:
            p_proc = filters.median(img_p, morphology.disk(2))
            p_proc = morphology.white_tophat(p_proc, morphology.disk(20))
            mask_p = p_proc > filters.threshold_li(p_proc)
            mask_p = morphology.remove_small_objects(mask_p, min_size=50)
        else:
            mask_p = img_p > filters.threshold_otsu(img_p)

        mask_p     = morphology.remove_small_objects(mask_p, min_size=100)
        mask_b_raw = img_b > filters.threshold_otsu(img_b)

    # ── Watershed on BODIPY - Perilipin ───────────────────────────────────────
    clean_mask = mask_b_raw & ~mask_p   # Perilipin lines act as separators
    distance   = ndi.distance_transform_edt(clean_mask)
    coords     = peak_local_max(distance, min_distance=10, labels=clean_mask)
    markers    = np.zeros_like(clean_mask, dtype=int)
    for i, (y, x) in enumerate(coords):
        markers[y, x] = i + 1
    label_b = watershed(-distance, markers, mask=clean_mask)
    label_b = morphology.remove_small_objects(label_b, min_size=MIN_AREA)

    # ── Adipocyte classification ───────────────────────────────────────────────
    dilated_p  = morphology.binary_dilation(mask_p, morphology.disk(2))
    label_ids, counts = np.unique(label_b[dilated_p], return_counts=True)
    touching_ids = label_ids[(label_ids > 0) & (counts > MIN_OVERLAP)]

    props   = measure.regionprops(label_b)
    records = []
    for prop in props:
        is_real_size   = MIN_AREA < prop.area < MAX_AREA
        is_on_membrane = prop.label in touching_ids
        records.append({
            'area':           prop.area,
            'is_adipocyte':   is_real_size and is_on_membrane,
            'is_real_size':   is_real_size,
            'is_on_membrane': is_on_membrane,
            'centroid':       prop.centroid,
        })

    # ── Visualization overlays ────────────────────────────────────────────────
    rainbow_labels = color.label2rgb(label_b, image=img_b, bg_label=0, alpha=0.3)
    valid_ids = [p.label for p in props
                 if p.label in touching_ids and MIN_AREA < p.area < MAX_AREA]
    overlay = np.zeros((*img_b.shape, 3))
    overlay[mask_b_raw]                  = [0.2, 0.2, 0.2]  # all lipid: dark grey
    overlay[np.isin(label_b, valid_ids)] = [0.0, 1.0, 0.0]  # adipocytes: green
    overlay[mask_p]                      = [1.0, 0.0, 1.0]  # Perilipin: magenta

    return records, label_b, overlay, rainbow_labels

In [ ]:
all_adipocyte_details = []

for s in SLIST:
    path = f"{QPTIFF_DIR}{s}_FITC_20x.qptiff"
    print(f"\n{'='*50}\nProcessing {s}...")

    with tff.TiffFile(path) as tif:
        image_he = tif.series[0].levels[0].asarray()

    img_bodipy = image_he[int(CH_ID_BODIPY), :, :]
    img_perl   = image_he[int(CH_ID_PERL),   :, :]

    df_sub = final_df.query('sample == @s')

    for secid in df_sub.section_id.unique():
        x0, xf, y0, yf = df_sub.query('section_id == @secid')[['x0','xf','y0','yf']].iloc[0]
        img_b = img_bodipy[y0:yf, x0:xf]
        img_p = img_perl[y0:yf, x0:xf]

        records, label_b, overlay, rainbow_labels = segment_section(img_b, img_p, s)

        for rec in records:
            rec.update({'sample': s, 'section_id': secid})
            all_adipocyte_details.append(rec)

        n_adip = sum(r['is_adipocyte'] for r in records)
        print(f"  {secid}: {len(records)} droplets found, {n_adip} classified as adipocytes")

        fig, ax = plt.subplots(1, 2, figsize=(20, 10))
        ax[0].imshow(rainbow_labels)
        ax[0].set_title("Individual Droplets (Watershed)")
        ax[1].imshow(overlay)
        ax[1].set_title("Confirmed Adipocytes (Size + Perilipin Filtered)")
        plt.suptitle(f"Green: Adipocytes | Magenta: Perilipin | {s}-{secid}")
        plt.savefig(f"{RESULTS_DIR}segmentation_{s}_{secid}.pdf", dpi=150, bbox_inches='tight')
        plt.show()

df_adipocyte_full = pd.DataFrame(all_adipocyte_details)
df_adipocyte_full['pid']  = df_adipocyte_full.section_id.map(DICT_CID2PID).astype(str)
df_adipocyte_full['Type'] = df_adipocyte_full.section_id.map(DICT_CID2TYPE)

df_adipocyte_full.to_csv(RESULTS_DIR + 'adipocyte_droplets_all.csv', index=False)
print(f"\nTotal droplets: {len(df_adipocyte_full):,}")
print(f"Adipocytes:     {df_adipocyte_full.is_adipocyte.sum():,}")

---
## 5. Tissue Area (from QuPath)

Tissue area segmentation was performed in [QuPath](https://qupath.github.io/) using pixel classifier thresholding on the BODIPY + DAPI composite, which gave more reliable boundaries than automated Python-based approaches for these slides.

Areas are in µm² and are converted to mm² for density calculations.

In [ ]:
# QuPath-derived adipose tissue areas per section (µm²)
dict_area = {
    'area_um2': [
        3610616.10, 5268363.72, 2927003.56, 3815687.09, 3540492.17,
        3165539.89, 2598992.17, 6145870.85, 1932623.22, 7890922.13, 8967752.38
    ],
    'slide': [
        'Slide1', 'Slide1', 'Slide2', 'Slide2', 'Slide3',
        'Slide4', 'Slide4', 'Slide5', 'Slide5', 'Slide6', 'Slide6'
    ],
    'section_id': [
        '1623464', '1622753', '1614806', '1622668', '1621945',
        '1623790', '1615778', '1629507', '1629501', '1631298', '1622930'
    ]
}

df_area = pd.DataFrame(dict_area)
df_area['tissue_area_mm2'] = df_area['area_um2'] / 1_000_000
print(df_area)

---
## 6. Lipid Droplet Size Distribution

Compare the size of confirmed adipocyte lipid droplets between Day1 and Day60. Areas are converted from pixels² to µm² using the known pixel size at 20×.

In [ ]:
# Load results (or use df_adipocyte_full if running end-to-end)
# df_adipocyte_full = pd.read_csv(RESULTS_DIR + 'adipocyte_droplets_all.csv', dtype={'section_id': str})

PIXEL_AREA_TO_UM2 = PIXEL_SIZE_UM ** 2   # 1 px² → µm²

df_true_adipocytes = df_adipocyte_full[df_adipocyte_full['is_adipocyte']].copy()
df_true_adipocytes['area_um2'] = df_true_adipocytes['area'] * PIXEL_AREA_TO_UM2
df_true_adipocytes['area_mm2'] = df_true_adipocytes['area_um2'] / 1_000_000

print(f"Adipocytes: {len(df_true_adipocytes):,} droplets across "
      f"{df_true_adipocytes['section_id'].nunique()} sections")
df_true_adipocytes.groupby('Type')['area_um2'].describe().round(1)

In [ ]:
# KDE of droplet area distribution by condition
fig, ax = plt.subplots(figsize=(5, 3))
sns.kdeplot(data=df_true_adipocytes, x='area_um2', hue='Type',
            fill=True, palette=DICT_COLOR, hue_order=HUE_ORDER, ax=ax)
ax.set_xlabel("Lipid Droplet Area (µm²)")
ax.set_ylabel("Density")
ax.set_title("Lipid Droplet Size Distribution by Condition")
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'kde_droplet_size_by_condition.pdf', dpi=300)
plt.show()

In [ ]:
# Boxplot: mean droplet area per section, Day1 vs Day60
fig, ax = plt.subplots(figsize=(3.5, 3))
sns.boxplot(x='Type', y='area_um2', data=df_true_adipocytes,
            order=HUE_ORDER, palette=DICT_COLOR, ax=ax, showfliers=False)

pairs = [("Day1", "Day60")]
annotator = Annotator(ax, pairs, data=df_true_adipocytes, x='Type', y='area_um2')
annotator.configure(test='t-test_ind', text_format='star', loc='outside')
annotator.apply_and_annotate()

ax.set_xlabel("")
ax.set_ylabel("Droplet Area (µm²)")
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'boxplot_droplet_size.pdf', dpi=300)
plt.show()

---
## 7. Adipocyte Count Density

Number of confirmed adipocyte lipid droplets per mm² of adipose tissue, comparing Day1 vs Day60.

In [ ]:
# Count droplets per section and merge with tissue area
df_counts = (df_true_adipocytes
             .groupby(['sample', 'section_id', 'Type', 'pid'])
             .size()
             .reset_index(name='cell_count'))

df_density = pd.merge(
    df_counts, df_area,
    left_on=['sample', 'section_id'],
    right_on=['slide', 'section_id'],
    how='inner'
)
df_density['density_mm2'] = df_density['cell_count'] / df_density['tissue_area_mm2']
print(df_density[['sample', 'section_id', 'Type', 'pid', 'cell_count', 'tissue_area_mm2', 'density_mm2']])

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
sns.boxplot(data=df_density, x='Type', y='density_mm2',
            palette=DICT_COLOR, showfliers=False, order=HUE_ORDER, ax=ax)
sns.stripplot(data=df_density, x='Type', y='density_mm2',
              hue='pid', dodge=False, s=5, linewidth=0, alpha=0.8,
              order=HUE_ORDER, ax=ax)

pairs = [("Day1", "Day60")]
annotator = Annotator(ax, pairs, data=df_density, x='Type', y='density_mm2')
annotator.configure(test='t-test_ind', text_format='star')
annotator.apply_and_annotate()

ax.set_xlabel("")
ax.set_ylabel("Adipocyte density (per mm²)")
ax.set_title("Adipocyte Count Density")
plt.legend(title='Patient ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'boxplot_droplet_count_density.pdf', dpi=300)
plt.show()

---
## 8. Lipid Area Fraction

Total lipid droplet area as a percentage of the adipose tissue area, comparing Day1 vs Day60.

In [ ]:
# Sum lipid pixel area per section and convert to µm²
df_lipid_sum = (df_true_adipocytes
                .groupby(['sample', 'section_id', 'Type', 'pid'])
                .agg(area=('area', 'sum'))
                .reset_index())

df_final = pd.merge(
    df_lipid_sum, df_area,
    left_on=['sample', 'section_id'],
    right_on=['slide', 'section_id'],
    how='inner'
)

df_final['lipid_area_um2']    = df_final['area'] * PIXEL_AREA_TO_UM2
df_final['area_density_pct']  = (df_final['lipid_area_um2'] / df_final['area_um2']) * 100

print(df_final[['sample', 'section_id', 'Type', 'pid', 'lipid_area_um2', 'area_um2', 'area_density_pct']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
sns.boxplot(data=df_final, x='Type', y='area_density_pct',
            palette=DICT_COLOR, showfliers=False, order=HUE_ORDER, ax=ax)
sns.stripplot(data=df_final, x='Type', y='area_density_pct',
              hue='pid', dodge=False, s=5, linewidth=0, alpha=0.8,
              order=HUE_ORDER, ax=ax)

pairs = [("Day1", "Day60")]
annotator = Annotator(ax, pairs, data=df_final, x='Type', y='area_density_pct')
annotator.configure(test='t-test_ind', text_format='star', loc='outside')
annotator.apply_and_annotate()

ax.set_xlabel("")
ax.set_ylabel("Lipid area fraction (%)")
plt.legend(title='Patient ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'boxplot_lipid_area_fraction.pdf', dpi=300)
plt.show()